In [1]:
import os
print("Current dir:", os.getcwd())

Current dir: C:\Users\skhul\Documents\Retail-work\documentation\06-database


In [4]:
readme = """# M07-01 — Define Database Entities and Data Requirements

## Purpose
Logical database schema for Retail-X-AI, based on:
- `dataset/raw/retail_store_inventory.csv` (73,100 rows × 15 columns)
- M02-04 Data Dictionary
- M02-05 Forecasting target & leakage risks

## Contents
| File | Description |
|------|-------------|
| `README.md` | This file |
| `erd_diagram.png` | Entity-Relationship Diagram (exported from dbdiagram.io) |
| `schema.sql` | CREATE TABLE statements + constraints + indexes |
| `data_requirements.md` | Full data requirements specification |
| `sample_queries.sql` | 8 analytical queries demonstrating usability |

## Entities Identified (6)
| # | Entity | Rows | Purpose |
|---|--------|------|---------|
| 1 | Category | 5 | Product groups |
| 2 | Region | 4 | Geographic territories |
| 3 | Store | 5 | Retail locations |
| 4 | Product | 20 | Products sold |
| 5 | Date_Dim | 731 | Daily calendar dimension |
| 6 | Inventory_Record | 73,100 | Daily fact table |

## Dependency Status
| Task | Status | Impact |
|------|--------|--------|
| M02-04 — Data Dictionary | ✅ Read | Types, ranges, quality issues confirmed |
| M02-05 — Forecasting target & leakage | ✅ Read | Target column identified |
| M04-01 — AI model design | ⏳ Pending | §3.7 feature refinement deferred |

## Known Data Quality Issues
- `Demand Forecast` contains **673 negative values** (min = -9.99).
  Must be cleaned to `>= 0` before ingestion. Enforced by CHECK constraint.
- Dataset is **synthetic** (Kaggle source, see `dataset/raw/DATASET-SOURCE.md`).
- `Holiday/Promotion` is the same column as `Promotion` in the Kaggle source.

## Assignee
@KhanyisileSkhulile614

## Status
- [x] Entities identified
- [x] ERD drawn (to be exported)
- [x] Schema written
- [x] Data requirements documented
- [x] Sample queries added
- [ ] PR merged
- [ ] Issue #33 closed
"""

with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme)

import os
print("✅ README.md written")
print("Size:", os.path.getsize("README.md"), "bytes")

✅ README.md written
Size: 1922 bytes


In [5]:
schema = """-- ============================================================
-- Retail-X-AI — Database Schema
-- M07-01: Define Database Entities and Data Requirements
-- Author:  @KhanyisileSkhulile614
-- Date:    2026-10-12
-- Based on: M02-04 Data Dictionary + retail_store_inventory.csv
-- ============================================================

DROP TABLE IF EXISTS Inventory_Record;
DROP TABLE IF EXISTS Product;
DROP TABLE IF EXISTS Store;
DROP TABLE IF EXISTS Date_Dim;
DROP TABLE IF EXISTS Category;
DROP TABLE IF EXISTS Region;

-- 1. Category
CREATE TABLE Category (
    Category_ID   INTEGER PRIMARY KEY AUTOINCREMENT,
    Category_Name VARCHAR(50) NOT NULL UNIQUE
);

-- 2. Region
CREATE TABLE Region (
    Region_ID   INTEGER PRIMARY KEY AUTOINCREMENT,
    Region_Name VARCHAR(50) NOT NULL UNIQUE
);

-- 3. Store
CREATE TABLE Store (
    Store_ID   VARCHAR(10)  PRIMARY KEY,
    Region_ID  INTEGER      NOT NULL,
    Store_Name VARCHAR(100) NOT NULL,
    FOREIGN KEY (Region_ID) REFERENCES Region(Region_ID)
);

-- 4. Product
CREATE TABLE Product (
    Product_ID   VARCHAR(10)   PRIMARY KEY,
    Category_ID  INTEGER       NOT NULL,
    Product_Name VARCHAR(100)  NOT NULL,
    Unit_Price   DECIMAL(10,2) NOT NULL CHECK (Unit_Price > 0),
    Discount     DECIMAL(5,2)  DEFAULT 0 CHECK (Discount BETWEEN 0 AND 100),
    FOREIGN KEY (Category_ID) REFERENCES Category(Category_ID)
);

-- 5. Date_Dim
CREATE TABLE Date_Dim (
    Date_ID   INTEGER     PRIMARY KEY AUTOINCREMENT,
    Full_Date DATE        NOT NULL UNIQUE,
    Day       INTEGER     NOT NULL CHECK (Day   BETWEEN 1 AND 31),
    Month     INTEGER     NOT NULL CHECK (Month BETWEEN 1 AND 12),
    Year      INTEGER     NOT NULL,
    Season    VARCHAR(20) NOT NULL
        CHECK (Season IN ('Spring','Summer','Autumn','Winter'))
);

-- 6. Inventory_Record (fact table)
CREATE TABLE Inventory_Record (
    Record_ID          INTEGER       PRIMARY KEY AUTOINCREMENT,
    Date_ID            INTEGER       NOT NULL,
    Store_ID           VARCHAR(10)   NOT NULL,
    Product_ID         VARCHAR(10)   NOT NULL,
    Inventory_Level    INTEGER       NOT NULL CHECK (Inventory_Level >= 0),
    Units_Sold         INTEGER       NOT NULL CHECK (Units_Sold >= 0),
    Units_Ordered      INTEGER       NOT NULL CHECK (Units_Ordered >= 0),
    Demand_Forecast    DECIMAL(10,2) NOT NULL CHECK (Demand_Forecast >= 0),
    Competitor_Pricing DECIMAL(10,2) CHECK (Competitor_Pricing > 0),
    Weather_Condition  VARCHAR(20)
        CHECK (Weather_Condition IN ('Rainy','Sunny','Cloudy','Snowy')),
    Holiday_Promotion  BOOLEAN DEFAULT 0,
    Seasonality        VARCHAR(20)
        CHECK (Seasonality IN ('Spring','Summer','Autumn','Winter')),
    UNIQUE (Date_ID, Store_ID, Product_ID),
    FOREIGN KEY (Date_ID)    REFERENCES Date_Dim(Date_ID),
    FOREIGN KEY (Store_ID)   REFERENCES Store(Store_ID),
    FOREIGN KEY (Product_ID) REFERENCES Product(Product_ID)
);

-- Indexes
CREATE INDEX idx_inv_date    ON Inventory_Record(Date_ID);
CREATE INDEX idx_inv_store   ON Inventory_Record(Store_ID);
CREATE INDEX idx_inv_product ON Inventory_Record(Product_ID);
CREATE INDEX idx_inv_weather ON Inventory_Record(Weather_Condition);
CREATE INDEX idx_inv_season  ON Inventory_Record(Seasonality);
CREATE INDEX idx_inv_promo   ON Inventory_Record(Holiday_Promotion);

-- Seed lookups
INSERT INTO Category (Category_Name) VALUES
    ('Groceries'), ('Toys'), ('Electronics'), ('Furniture'), ('Clothing');
INSERT INTO Region (Region_Name) VALUES
    ('North'), ('South'), ('East'), ('West');
INSERT INTO Store (Store_ID, Region_ID, Store_Name) VALUES
    ('S001', 1, 'Store S001'),
    ('S002', 2, 'Store S002'),
    ('S003', 3, 'Store S003'),
    ('S004', 4, 'Store S004'),
    ('S005', 1, 'Store S005');
"""

with open("schema.sql", "w", encoding="utf-8") as f:
    f.write(schema)

import os
print(" schema.sql written")
print("Size:", os.path.getsize("schema.sql"), "bytes")

 schema.sql written
Size: 3844 bytes


In [6]:
import sqlite3, os

# Fresh test DB
if os.path.exists("test.db"):
    os.remove("test.db")

conn = sqlite3.connect("test.db")
cur = conn.cursor()

with open("schema.sql", "r", encoding="utf-8") as f:
    cur.executescript(f.read())

conn.commit()

cur.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
tables = [r[0] for r in cur.fetchall()]
print("Tables created:", tables)

for t in ["Category", "Region", "Store"]:
    cur.execute(f"SELECT COUNT(*) FROM {t}")
    print(f"  {t}: {cur.fetchone()[0]} rows")

# Test a CHECK constraint (should fail)
try:
    cur.execute("INSERT INTO Inventory_Record (Date_ID, Store_ID, Product_ID, Inventory_Level, Units_Sold, Units_Ordered, Demand_Forecast) VALUES (999, 'S001', 'P0001', -5, 10, 10, 5.0)")
    print("⚠ CHECK constraint did NOT trigger (should not happen)")
except sqlite3.IntegrityError as e:
    print(" CHECK constraint works:", e)

conn.close()
print("\n Schema test complete")

Tables created: ['Category', 'Date_Dim', 'Inventory_Record', 'Product', 'Region', 'Store', 'sqlite_sequence']
  Category: 5 rows
  Region: 4 rows
  Store: 5 rows
 CHECK constraint works: CHECK constraint failed: Inventory_Level >= 0

 Schema test complete


In [7]:
reqs = """# Data Requirements — Retail-X-AI

**Task:** M07-01
**Author:** @KhanyisileSkhulile614
**Date:** 2026-10-12
**Based on:** M02-04 Data Dictionary, `dataset/raw/retail_store_inventory.csv`

---

## 1. Data Source

| Attribute | Value |
|-----------|-------|
| Source file | `dataset/raw/retail_store_inventory.csv` |
| Rows | 73,100 |
| Columns | 15 |
| Granularity | One row per (Date, Store ID, Product ID) |
| Date range | 2022-01-01 to 2022-09-30 (731 unique days) |
| Nature | Synthetic (Kaggle source, see DATASET-SOURCE.md) |

---

## 2. Entities and Row Counts

| # | Entity | Purpose | Rows |
|---|--------|---------|------|
| 1 | Category | Product groups | 5 |
| 2 | Region | Geographic territories | 4 |
| 3 | Store | Retail locations | 5 |
| 4 | Product | Products sold | 20 |
| 5 | Date_Dim | Daily calendar dimension | 731 |
| 6 | Inventory_Record | Daily fact table | 73,100 |

---

## 3. Field-Level Requirements

### 3.1 Category
| Field | Type | Constraints |
|-------|------|-------------|
| Category_ID | INT | PK, auto |
| Category_Name | VARCHAR(50) | UNIQUE, NOT NULL |

### 3.2 Region
| Field | Type | Constraints |
|-------|------|-------------|
| Region_ID | INT | PK, auto |
| Region_Name | VARCHAR(50) | UNIQUE, NOT NULL |

### 3.3 Store
| Field | Type | Constraints |
|-------|------|-------------|
| Store_ID | VARCHAR(10) | PK |
| Region_ID | INT | FK to Region |
| Store_Name | VARCHAR(100) | NOT NULL |

### 3.4 Product
| Field | Type | Constraints |
|-------|------|-------------|
| Product_ID | VARCHAR(10) | PK |
| Category_ID | INT | FK to Category |
| Product_Name | VARCHAR(100) | NOT NULL |
| Unit_Price | DECIMAL(10,2) | > 0 |
| Discount | DECIMAL(5,2) | 0 to 100, default 0 |

### 3.5 Date_Dim
| Field | Type | Constraints |
|-------|------|-------------|
| Date_ID | INT | PK, auto |
| Full_Date | DATE | UNIQUE, NOT NULL |
| Day | INT | 1 to 31 |
| Month | INT | 1 to 12 |
| Year | INT | > 0 |
| Season | VARCHAR(20) | IN (Spring, Summer, Autumn, Winter) |

### 3.6 Inventory_Record (fact table)
| Field | Type | Constraints | Notes |
|-------|------|-------------|-------|
| Record_ID | BIGINT | PK, auto | |
| Date_ID | INT | FK to Date_Dim | |
| Store_ID | VARCHAR(10) | FK to Store | |
| Product_ID | VARCHAR(10) | FK to Product | |
| Inventory_Level | INT | >= 0 | Observed 50 to 500 |
| Units_Sold | INT | >= 0 | Observed 0 to 499 |
| Units_Ordered | INT | >= 0 | Observed 20 to 200 |
| Demand_Forecast | DECIMAL(10,2) | >= 0 | 673 negatives in CSV - clean first |
| Competitor_Pricing | DECIMAL(10,2) | > 0 | Observed 5.03 to 104.94 |
| Weather_Condition | VARCHAR(20) | IN (Rainy, Sunny, Cloudy, Snowy) | |
| Holiday_Promotion | BOOLEAN | 0 or 1 | |
| Seasonality | VARCHAR(20) | IN (Spring, Summer, Autumn, Winter) | |

### 3.7 Feature Columns for Modelling - PENDING M04-01

To be finalised once M04-01 (AI model design) is available.
Placeholder based on M02-05 findings:

- **Target variable:** [Demand_Forecast OR Units_Sold - confirm from M02-05]
- **Leakage column (exclude from inputs):** [the other one]

**Candidate input features:**
- Inventory_Level
- Units_Ordered
- Price
- Discount
- Weather_Condition
- Holiday_Promotion
- Competitor_Pricing
- Seasonality

**Derived features (to confirm):**
- lag_7_Units_Sold, lag_30_Units_Sold
- rolling_avg_7_Units_Sold, rolling_avg_30_Units_Sold

**Split strategy:** temporal (earlier months train, later months test)

---

## 4. Data Quality Rules

| # | Rule | Field(s) | Action if Violated |
|---|------|----------|--------------------|
| 1 | No missing values | All | Reject row |
| 2 | Non-negative stock | Inventory_Level | Clamp to 0 |
| 3 | Non-negative sales | Units_Sold | Reject row |
| 4 | Non-negative demand | Demand_Forecast | Replace 673 negatives with 0 before load |
| 5 | Positive price | Unit_Price, Competitor_Pricing | Reject row |
| 6 | Discount range | Discount | Clamp to [0, 100] |
| 7 | Valid categories | Category, Region, Weather, Seasonality | Reject row |
| 8 | Unique composite | (Date_ID, Store_ID, Product_ID) | Reject duplicate |
| 9 | FK integrity | All FKs | Reject row if parent missing |

---

## 5. Data Volume and Retention

- Daily load: 100 rows (5 stores x 20 products)
- Yearly load: ~36,500 rows
- Retention: minimum 24 months
- Backup: nightly snapshot to cloud storage

---

## 6. Indexing Strategy

| Index | Table | Column(s) | Purpose |
|-------|-------|-----------|---------|
| idx_inv_date | Inventory_Record | Date_ID | Time-series queries |
| idx_inv_store | Inventory_Record | Store_ID | Store-level analysis |
| idx_inv_product | Inventory_Record | Product_ID | Product-level analysis |
| idx_inv_weather | Inventory_Record | Weather_Condition | Weather impact |
| idx_inv_season | Inventory_Record | Seasonality | Seasonal analysis |
| idx_inv_promo | Inventory_Record | Holiday_Promotion | Promotion impact |

---

## 7. Dependencies

- M02-04 - Data Dictionary: column types, ranges, quality issues confirmed
- M02-05 - Forecasting target and leakage: identifies target variable
- M04-01 - AI model design: feature columns will be confirmed once available

---

## 8. Known Issues

1. Demand Forecast has 673 negative values (min = -9.99). Must be cleaned to >= 0
   before ingestion. Enforced by CHECK constraint in schema.sql.
2. Dataset is synthetic (not real business data).
3. Holiday/Promotion corresponds to Promotion in the original Kaggle source.
"""

with open("data_requirements.md", "w", encoding="utf-8") as f:
    f.write(reqs)

import os
print(" data_requirements.md written")
print("Size:", os.path.getsize("data_requirements.md"), "bytes")

 data_requirements.md written
Size: 5592 bytes


In [8]:
queries = """-- ============================================================
-- Retail-X-AI - Sample Queries
-- M07-01 - demonstrate schema usability
-- ============================================================

-- Q1. Total units sold per store
SELECT s.Store_ID,
       SUM(i.Units_Sold) AS Total_Units_Sold
FROM Inventory_Record i
JOIN Store s ON i.Store_ID = s.Store_ID
GROUP BY s.Store_ID
ORDER BY Total_Units_Sold DESC;

-- Q2. Top 5 best-selling products
SELECT p.Product_ID,
       p.Product_Name,
       SUM(i.Units_Sold) AS Total_Sold
FROM Inventory_Record i
JOIN Product p ON i.Product_ID = p.Product_ID
GROUP BY p.Product_ID, p.Product_Name
ORDER BY Total_Sold DESC
LIMIT 5;

-- Q3. Sales by category
SELECT c.Category_Name,
       SUM(i.Units_Sold) AS Total_Sold,
       ROUND(AVG(i.Units_Sold), 2) AS Avg_Daily_Sales
FROM Inventory_Record i
JOIN Product p ON i.Product_ID = p.Product_ID
JOIN Category c ON p.Category_ID = c.Category_ID
GROUP BY c.Category_Name
ORDER BY Total_Sold DESC;

-- Q4. Weather impact on sales
SELECT Weather_Condition,
       COUNT(*) AS Days,
       ROUND(AVG(Units_Sold), 2) AS Avg_Units_Sold
FROM Inventory_Record
GROUP BY Weather_Condition
ORDER BY Avg_Units_Sold DESC;

-- Q5. Holiday vs non-holiday sales
SELECT Holiday_Promotion,
       COUNT(*) AS Records,
       ROUND(AVG(Units_Sold), 2) AS Avg_Units_Sold
FROM Inventory_Record
GROUP BY Holiday_Promotion;

-- Q6. Demand forecast accuracy (MAE and MAPE)
SELECT
    ROUND(AVG(ABS(Units_Sold - Demand_Forecast)), 2) AS MAE,
    ROUND(100 * AVG(ABS(Units_Sold - Demand_Forecast)
        / NULLIF(Units_Sold, 0)), 2) AS MAPE_Percent
FROM Inventory_Record;

-- Q7. Seasonal sales pattern
SELECT d.Season,
       SUM(i.Units_Sold) AS Total_Sold
FROM Inventory_Record i
JOIN Date_Dim d ON i.Date_ID = d.Date_ID
GROUP BY d.Season
ORDER BY Total_Sold DESC;

-- Q8. Inventory turnover by store
SELECT s.Store_ID,
       ROUND(SUM(i.Units_Sold) * 1.0 / NULLIF(SUM(i.Inventory_Level), 0), 4)
           AS Turnover_Ratio
FROM Inventory_Record i
JOIN Store s ON i.Store_ID = s.Store_ID
GROUP BY s.Store_ID
ORDER BY Turnover_Ratio DESC;
"""

with open("sample_queries.sql", "w", encoding="utf-8") as f:
    f.write(queries)

import os
print(" sample_queries.sql written")
print("Size:", os.path.getsize("sample_queries.sql"), "bytes")

 sample_queries.sql written
Size: 2184 bytes


In [9]:
import os

print("Files in current folder:")
print("-" * 50)
for f in sorted(os.listdir(".")):
    if os.path.isfile(f):
        print(f"  {f:35s} {os.path.getsize(f):>8,} bytes")
print("-" * 50)

Files in current folder:
--------------------------------------------------
  .gitkeep                                   0 bytes
  README.md                              1,922 bytes
  data_requirements.md                   5,592 bytes
  m07_build.ipynb                       18,580 bytes
  sample_queries.sql                     2,184 bytes
  schema.sql                             3,844 bytes
  test.db                               81,920 bytes
--------------------------------------------------


In [10]:
import sqlite3

conn = sqlite3.connect("test.db")
cur = conn.cursor()

query_list = queries.split(";")
executed = 0

for q in query_list:
    q = q.strip()
    if not q or q.startswith("--"):
        continue
    try:
        cur.execute(q)
        cur.fetchall()
        executed += 1
    except Exception as e:
        print(f"❌ FAILED: {q[:60]}...")
        print(f"   Error: {e}")

print(f"\n✅ {executed} queries executed without syntax errors")
conn.close()


✅ 0 queries executed without syntax errors


In [11]:
import os

# Remove test DB
if os.path.exists("test.db"):
    os.remove("test.db")
    print("✅ test.db removed")
else:
    print("test.db not found")

# Write .gitignore
gitignore = """# Temporary test files
*.db
*.sqlite
*.sqlite3

# Jupyter checkpoints
.ipynb_checkpoints/

# OS files
.DS_Store
Thumbs.db
"""

with open(".gitignore", "w", encoding="utf-8") as f:
    f.write(gitignore)

print("✅ .gitignore written")

✅ test.db removed
✅ .gitignore written


In [12]:
# Diagnostic
print("Type of queries:", type(queries))
print("Length of queries string:", len(queries))
print("First 200 chars:")
print(repr(queries[:200]))

Type of queries: <class 'str'>
Length of queries string: 2113
First 200 chars:
'-- ============================================================\n-- Retail-X-AI - Sample Queries\n-- M07-01 - demonstrate schema usability\n-- ============================================================'


In [13]:
import sqlite3, re, os

# Recreate test.db (we deleted it in Cell 9)
if os.path.exists("test.db"):
    os.remove("test.db")

conn = sqlite3.connect("test.db")
cur = conn.cursor()

with open("schema.sql", "r", encoding="utf-8") as f:
    cur.executescript(f.read())
conn.commit()

# Strip line comments, then split on semicolons
cleaned = re.sub(r"--[^\n]*", "", queries)
statements = [s.strip() for s in cleaned.split(";") if s.strip()]

print(f"Found {len(statements)} statements\n")

executed = 0
for i, stmt in enumerate(statements, 1):
    try:
        cur.execute(stmt)
        cur.fetchall()
        executed += 1
        print(f"  ✅ Q{i}: {stmt[:50].strip()}...")
    except Exception as e:
        print(f"  ❌ Q{i}: {stmt[:50].strip()}...")
        print(f"     Error: {e}")

print(f"\n✅ {executed} / {len(statements)} queries executed successfully")
conn.close()

Found 8 statements

  ✅ Q1: SELECT s.Store_ID,
       SUM(i.Units_Sold) AS Tot...
  ✅ Q2: SELECT p.Product_ID,
       p.Product_Name,...
  ✅ Q3: SELECT c.Category_Name,
       SUM(i.Units_Sold) A...
  ✅ Q4: SELECT Weather_Condition,
       COUNT(*) AS Days,...
  ✅ Q5: SELECT Holiday_Promotion,
       COUNT(*) AS Recor...
  ✅ Q6: SELECT
    ROUND(AVG(ABS(Units_Sold - Demand_Forec...
  ✅ Q7: SELECT d.Season,
       SUM(i.Units_Sold) AS Total...
  ✅ Q8: SELECT s.Store_ID,
       ROUND(SUM(i.Units_Sold)...

✅ 8 / 8 queries executed successfully


In [14]:
import os
if os.path.exists("test.db"):
    os.remove("test.db")
    print("✅ test.db removed")
else:
    print("test.db not found")

print("\nFinal files in folder:")
for f in sorted(os.listdir(".")):
    if os.path.isfile(f):
        print(f"  {f:30s} {os.path.getsize(f):>8,} bytes")

✅ test.db removed

Final files in folder:
  .gitignore                          133 bytes
  .gitkeep                              0 bytes
  README.md                         1,922 bytes
  data_requirements.md              5,592 bytes
  m07_build.ipynb                  25,885 bytes
  sample_queries.sql                2,184 bytes
  schema.sql                        3,844 bytes
